# High-throughput targeted amplicon screening tool for characterizing intrahost diversity in *Staphylococcus aureus* directly from sample 

Creating a minimum PMO from the data downloaded from the following study:

Furstenau, T. N., Whealy, R., Timm, S., Roberts, A., Maltinsky, S., Wells, S. J., Drake, K., Ross, A., Bolduc, C., Pearson, T., & Fofanov, V. Y. (2025). *High-throughput targeted amplicon screening tool for characterizing intrahost diversity in* Staphylococcus aureus *directly from sample*. Microbial Genomics, 11(6). https://doi.org/10.1099/mgen.0.001427

In [29]:
import pandas as pd
from pmotools.pmo_builder.panel_information_to_pmo import panel_info_table_to_pmo, merge_panel_info_dicts
from pmotools.pmo_builder.metatable_to_pmo import library_sample_info_table_to_pmo, specimen_info_table_to_pmo
from pmotools.pmo_builder.mhap_table_to_pmo import (
    mhap_table_to_pmo, 
    create_minimum_library_specimen_dict_from_mhap_table)
from pmotools.pmo_builder.merge_to_pmo import merge_to_pmo
from pmotools.pmo_engine.pmo_writer import * 
from pmotools.pmo_engine.pmo_checker import PMOChecker
import numpy as np
from pmotools.pmo_builder import panel_information_to_pmo

## Required data

The minimum amount of information needed to create a PMO is the **microhaplotype** data and information on the **panel** used (at a minimum, the **target's primers**)

* allele_data.tsv.gz - results of microhaplotype called data
* Furstenau2025_primers.tsv - primers used in the experiment

## Merging info pmo 


In [30]:
# read in data 
mhap_info_df = pd.read_csv("allele_data.tsv.gz", sep='\t')
primers = pd.read_csv("Furstenau2025_primers.tsv", sep='\t')

# convert panel info
pmo_panel_and_target_info = panel_info_table_to_pmo(primers, 
                                                    panel_name = "staph_aureus_Furstenau2025",
                                                    target_name_col = "target",
                                                    forward_primers_seq_col = "forward", 
                                                    reverse_primers_seq_col = "reverse")
# convert microhaplotype data
pmo_mhaps = mhap_table_to_pmo(
                       microhaplotype_table=mhap_info_df, 
                       library_sample_name_col='s_Sample',
                       target_name_col='p_name',
                       seq_col='h_Consensus',
                       reads_col='c_ReadCnt')

# merge into pmo
staph_aureus_pmo = merge_to_pmo(
    panel_target_info = pmo_panel_and_target_info,
    mhap_info = pmo_mhaps
)

In [31]:
# Validate the PMO file against schema 
checker = PMOChecker()
checker.validate_pmo_json(staph_aureus_pmo)

In [32]:
# write out
pmowriter = PMOWriter()
pmowriter.write_out_pmo(staph_aureus_pmo, "minimum_Furstenau2025_PMO.json.gz", overwrite=True)

## Adding a key to set a specimen_name for library_sample_name 

The specimen_names and library_sample_names are auto generated from microhaplotype data with specimens being exactly the libary_sample_names. This can be changed by supply a table that supplies a specimen_name to be used for each library_sample_name. Can use the SRA meta info for this

* sra_info_table.tsv - this has the SRA/ENA meta information

List out the current specimen_name and library_sample_name to see how they are set up as default

In [33]:
from pmotools.pmo_engine.pmo_exporter import PMOExporter
lib_to_spec_df = PMOExporter.list_library_sample_names_per_specimen_name(staph_aureus_pmo)
lib_to_spec_df.head()

,specimen_name,library_sample_name,library_sample_count
0,SRR30825770,SRR30825770,1
1,SRR30825771,SRR30825771,1
2,SRR30825772,SRR30825772,1
3,SRR30825773,SRR30825773,1
4,SRR30825774,SRR30825774,1


Read in the SRA information which can be used to create a key to change the specimen_names

In [34]:
sra_info = pd.read_csv("sra_info_table.tsv", sep = '\t')
# create a dictionary key
lib_to_spec_key = sra_info.set_index('run_accession')['sample_alias'].to_dict()

# supply key when building library_sample_info and specimen_info 
library_sample_and_spec_renamed_infos = create_minimum_library_specimen_dict_from_mhap_table(
    pmo_mhaps["detected_microhaplotypes"], 
    panel_name = "staph_aureus_Furstenau2025", 
    library_sample_specimen_key = lib_to_spec_key)

# now build with renamed 
staph_aureus_pmo_renamed = merge_to_pmo(
    specimen_info = library_sample_and_spec_renamed_infos["specimen_info"],
    library_sample_info = library_sample_and_spec_renamed_infos["library_sample_info"],
    panel_target_info = pmo_panel_and_target_info,
    mhap_info = pmo_mhaps
)

lib_to_spec_renamed_df = PMOExporter.list_library_sample_names_per_specimen_name(staph_aureus_pmo_renamed)
lib_to_spec_renamed_df.head()

,specimen_name,library_sample_name,library_sample_count
0,85b498-Wk16-Nasal,SRR30825770,1
1,85b498-Wk28-Nasal,SRR30825771,1
2,85b498-Wk12-Nasal,SRR30825772,1
3,85b498-Wk20-Nasal,SRR30825773,1
4,85b498-Wk14-Nasal,SRR30825774,1


In [35]:
pmowriter.write_out_pmo(staph_aureus_pmo_renamed, "minimum_Furstenau2025_new_names_PMO.json.gz", overwrite=True)